# Ch 6 — Cliff Walking: SARSA vs Q-learning

**复现 Sutton & Barto 图 6.4** + 数值演示 SARSA 跟 Q-learning 学到不同 policy。

## Setup
- 4×12 grid world
- 起点 (3, 0)，终点 (3, 11)
- 底行 (3, 1)~(3, 10) 是 cliff，掉下去 reward = -100 + 回起点
- 其他每步 reward = -1
- γ = 1（episodic），ε-greedy 选 action

## 学什么
1. **Q-learning 学最优 Q*（沿悬崖边走）**
2. **SARSA 学当前 ε-greedy 策略的 Q（绕远路安全走）**
3. 在 ε > 0 部署下 SARSA 实际 sum reward 更高（不会乱摔）

## 链回
- [[ch06-temporal-difference-learning]] §3.3 Q-learning
- [[_anki/ch06-td-cards]] §C

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(0)

# Grid setup
ROWS, COLS = 4, 12
START = (3, 0)
GOAL = (3, 11)
CLIFF = [(3, c) for c in range(1, 11)]

# Actions: 0=up, 1=down, 2=left, 3=right
ACTIONS = [(-1, 0), (1, 0), (0, -1), (0, 1)]
N_ACTIONS = 4

def step(state, action):
    r, c = state
    dr, dc = ACTIONS[action]
    nr, nc = max(0, min(ROWS-1, r+dr)), max(0, min(COLS-1, c+dc))
    next_state = (nr, nc)
    if next_state in CLIFF:
        return START, -100, False  # fall in cliff, reset to start
    elif next_state == GOAL:
        return next_state, -1, True  # reach goal
    else:
        return next_state, -1, False

def eps_greedy(Q, state, eps):
    if np.random.rand() < eps:
        return np.random.randint(N_ACTIONS)
    return int(np.argmax(Q[state]))

## SARSA 实现

**Update**: Q(s, a) ← Q(s, a) + α · [r + γQ(s', a') - Q(s, a)]，A' 由当前 ε-greedy 选。

In [ ]:
def sarsa(n_episodes, alpha=0.5, eps=0.1, gamma=1.0):
    Q = np.zeros((ROWS, COLS, N_ACTIONS))
    rewards = []
    for _ in range(n_episodes):
        s = START
        a = eps_greedy(Q, s, eps)
        total_r = 0
        while True:
            s_next, r, done = step(s, a)
            total_r += r
            if done:
                Q[s][a] += alpha * (r - Q[s][a])
                break
            a_next = eps_greedy(Q, s_next, eps)
            Q[s][a] += alpha * (r + gamma * Q[s_next][a_next] - Q[s][a])
            s, a = s_next, a_next
        rewards.append(max(total_r, -100))  # cap for plotting
    return Q, rewards

Q_sarsa, r_sarsa = sarsa(500)
print(f'SARSA: final 100-ep avg reward = {np.mean(r_sarsa[-100:]):.1f}')

## Q-learning 实现

**Update**: Q(s, a) ← Q(s, a) + α · [r + γ · max_a' Q(s', a') - Q(s, a)]——off-policy。

In [ ]:
def qlearning(n_episodes, alpha=0.5, eps=0.1, gamma=1.0):
    Q = np.zeros((ROWS, COLS, N_ACTIONS))
    rewards = []
    for _ in range(n_episodes):
        s = START
        total_r = 0
        while True:
            a = eps_greedy(Q, s, eps)
            s_next, r, done = step(s, a)
            total_r += r
            if done:
                Q[s][a] += alpha * (r - Q[s][a])
                break
            # target uses max (greedy of Q), not eps-greedy
            Q[s][a] += alpha * (r + gamma * np.max(Q[s_next]) - Q[s][a])
            s = s_next
        rewards.append(max(total_r, -100))
    return Q, rewards

Q_q, r_q = qlearning(500)
print(f'Q-learning: final 100-ep avg reward = {np.mean(r_q[-100:]):.1f}')

## 比较 reward 曲线（复现 Sutton 图 6.4）

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
def smooth(x, w=20):
    return np.convolve(x, np.ones(w)/w, mode='valid')
ax.plot(smooth(r_sarsa), label='SARSA (on-policy)', color='C0')
ax.plot(smooth(r_q), label='Q-learning (off-policy)', color='C1')
ax.set_xlabel('Episode')
ax.set_ylabel('Sum reward per episode (smoothed)')
ax.set_title('Cliff Walking: SARSA vs Q-learning')
ax.set_ylim(-100, 0)
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 可视化学到的 policy

用箭头 print 每个格子的 greedy action。

In [ ]:
ARROWS = ['↑', '↓', '←', '→']

def print_policy(Q, name):
    print(f'\n=== {name} learned greedy policy ===')
    for r in range(ROWS):
        row = ''
        for c in range(COLS):
            if (r, c) == START:
                row += ' S '
            elif (r, c) == GOAL:
                row += ' G '
            elif (r, c) in CLIFF:
                row += ' X '
            else:
                row += f' {ARROWS[np.argmax(Q[r, c])]} '
        print(row)

print_policy(Q_sarsa, 'SARSA')
print_policy(Q_q, 'Q-learning')

## 关键观察

1. **Q-learning** 学到 **最优 Q*** → policy 沿悬崖边走（最短路）
2. **SARSA** 学到 **当前 ε-greedy policy 的 Q** → 远离悬崖（因为 ε 探索时会摔）
3. **训练时 SARSA reward 更高**（不会因 ε 探索踩悬崖）；Q-learning 真实部署（ε=0）reward 才是最优

## Interview 启示

- 经典 "为什么 SARSA 跟 Q-learning 学到不同 policy" 必考——能讲清 cliff walking 故事 = 懂 on-policy vs off-policy 的精髓
- 实战 deploy：如果 deploy 时还有 ε（如 robot 防止过拟合）→ 用 SARSA；deploy 完全 greedy → 用 Q-learning
- 这就是 RLHF 用 PPO（on-policy）的部分原因——LLM deploy 时 sampling 仍有温度，跟 SARSA 一样需要 on-policy 评估

## 链回
- [[ch06-temporal-difference-learning]] §3.2 SARSA / §3.3 Q-learning
- [[_anki/ch06-td-cards]] §C / §G Card 30